# I-Beam Design with Bayesian Optimization

You have a set of beams your instructor tested. Your job: use a Gaussian Process model
to predict which untested beam would be the strongest, then pick one to test.

You will make choices about how to set up the model. Those choices matter. You are graded
on your reasoning, not on whether your beam wins.

In [ ]:
!pip install -q scikit-learn scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF, ConstantKernel
from scipy.optimize import minimize_scalar
from scipy.stats import norm
import warnings
warnings.filterwarnings("ignore")

# Beam geometry constants
TOTAL_HEIGHT = 25.0   # mm, total beam height (web + 2 flanges)
B_FIXED = 16.0        # mm, flange width (constant across all beams)
LENGTH_M = 0.2023     # m, span length
YIELD_STRENGTH = 76e6 # Pa
E_MODULUS = 2.5e9     # Pa
G_MODULUS = E_MODULUS / 2.6
C1_3PT = 1.35         # moment coefficient for 3-point bending
MATERIAL_DENSITY = 1240  # kg/m^3

# --- Section property calculations ---

def _fillet_props(r_m):
    if r_m <= 0:
        return 0.0, 0.0
    A_f = r_m**2 * (4 - np.pi) / 4
    c_f = 2 * r_m / (3 * (4 - np.pi))
    return A_f, c_f

def calc_Ix(H, h, B, b, r=0):
    H_m, h_m, B_m, b_m, r_m = H/1e3, h/1e3, B/1e3, b/1e3, r/1e3
    Ix = (H_m**3 * b_m) / 12 + 2 * ((h_m**3 * B_m) / 12 + h_m * B_m * ((H_m + h_m) / 2)**2)
    A_f, c_f = _fillet_props(r_m)
    if A_f > 0:
        d_x = H_m / 2 + c_f
        Il = r_m**4 * (16 - 3 * np.pi) / 48
        Ix += 4 * (Il - A_f * c_f**2 + A_f * d_x**2)
    return Ix

def calc_Iy(H, h, B, b, r=0):
    H_m, h_m, B_m, b_m, r_m = H/1e3, h/1e3, B/1e3, b/1e3, r/1e3
    Iy = (H_m * b_m**3) / 12 + 2 * (h_m * B_m**3) / 12
    A_f, c_f = _fillet_props(r_m)
    if A_f > 0:
        d_y = b_m / 2 + c_f
        Il = r_m**4 * (16 - 3 * np.pi) / 48
        Iy += 4 * (Il - A_f * c_f**2 + A_f * d_y**2)
    return Iy

def calc_J(H, h, B, b, r=0):
    H_m, h_m, B_m, b_m, r_m = H/1e3, h/1e3, B/1e3, b/1e3, r/1e3
    J = (H_m * b_m**3 + 2 * B_m * h_m**3) / 3
    if r_m > 0:
        J += 4 * 0.15 * r_m**4
    return J

def calc_mass(H, h, B, b, r=0):
    H_m, h_m, B_m, b_m, r_m = H/1e3, h/1e3, B/1e3, b/1e3, r/1e3
    A = H_m * b_m + 2 * h_m * B_m
    A_f, _ = _fillet_props(r_m)
    return MATERIAL_DENSITY * LENGTH_M * (A + 4 * A_f) * 1000

def calc_str_w(H, B, b, r=0):
    h = (TOTAL_HEIGHT - H) / 2.0
    Ix = calc_Ix(H, h, B, b, r)
    strength = (4 * YIELD_STRENGTH * Ix) / (0.0125 * LENGTH_M)
    mass = calc_mass(H, h, B, b, r)
    if mass <= 0:
        return 0.0
    return strength / mass

def find_H_opt(b, B=B_FIXED, r=0):
    def obj(H):
        if H < 12.0 or H > 23.4: return 1e10
        h = (TOTAL_HEIGHT - H) / 2.0
        if h < 0 or h > 6.5: return 1e10
        return -calc_str_w(H, B, b, r)
    return minimize_scalar(obj, bounds=(12.0, 23.4), method='bounded').x

def stability_ratio(H, h, B, b, r=0):
    Iy = calc_Iy(H, h, B, b, r)
    J = calc_J(H, h, B, b, r)
    if Iy <= 0 or J <= 0: return 0.0
    Mcr = (C1_3PT * np.pi / LENGTH_M) * np.sqrt(E_MODULUS * Iy * G_MODULUS * J)
    Ix = calc_Ix(H, h, B, b, r)
    y_max = (H / 1e3 + h / 1e3)
    if y_max <= 0: return 0.0
    My = YIELD_STRENGTH * Ix / y_max
    if My <= 0: return 0.0
    return Mcr / My

def bH_to_R(b, H):
    h = (TOTAL_HEIGHT - H) / 2.0
    return stability_ratio(H, h, B_FIXED, b) if h > 0 else 0.0

def bH_to_dH(b, H):
    return H - find_H_opt(b)

# --- GP helpers ---

def normalize(X, bounds):
    Xn = X.copy().astype(float)
    for i, (lo, hi) in enumerate(bounds):
        Xn[:, i] = (X[:, i] - lo) / (hi - lo)
    return Xn

def train_gp(X, y, bounds, alpha=1e-3, kernel_type='matern'):
    Xn = normalize(X, bounds)
    y_log = np.log(y)
    y_mean = np.mean(y_log)
    y_c = y_log - y_mean
    nd = X.shape[1]
    if kernel_type == 'matern':
        base = Matern(length_scale=[0.5]*nd, length_scale_bounds=(0.1, 5.0), nu=2.5)
    else:
        base = RBF(length_scale=[0.5]*nd, length_scale_bounds=(0.1, 5.0))
    kernel = ConstantKernel(1.0) * base
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=15,
                                  alpha=alpha, normalize_y=False)
    gp.fit(Xn, y_c)
    return gp, Xn, y_c, y_mean

def predict_grid(gp, y_mean, bounds, resolution=80):
    g0 = np.linspace(0, 1, resolution)
    g1 = np.linspace(0, 1, resolution)
    G0, G1 = np.meshgrid(g0, g1)
    Xg = np.column_stack([G0.ravel(), G1.ravel()])
    if len(bounds) == 3:
        # For 3D parameterization, we need to handle differently
        # This is only used for 2D (b,H) surface plots
        pass
    mu, sig = gp.predict(Xg, return_std=True)
    mu_real = np.exp(mu + y_mean)
    return G0, G1, mu_real.reshape(G0.shape), sig.reshape(G0.shape)

def get_noise_variance(R):
    sigma_base = 0.001
    sigma_peak = 0.004
    width = 0.5
    log_R = np.log(max(R, 1e-6))
    bump = np.exp(-0.5 * (log_R / width)**2)
    return (sigma_base + sigma_peak * bump)**2

print("Setup complete.")

## Instructor Configuration

Your instructor will give you a data URL and noise bounds. Enter them below.

In [ ]:
# === GET THESE FROM YOUR INSTRUCTOR ===
DATA_URL = "https://raw.githubusercontent.com/andrewvoss8-boop/core-me-data-science-activities-public/main/data/I_beam_data_2var.csv"
NOISE_LO = 3e-5    # noise floor (from duplicate beam tests)
NOISE_HI = 3e-3    # upper bound for noise parameter
N_BEAMS_TO_PICK = 1 # how many beams you get to recommend

# Load the data
df = pd.read_csv(DATA_URL)
b_col = [c for c in df.columns if 'b' in c.lower() and 'web' in c.lower()][0]
H_col = [c for c in df.columns if 'h' in c.lower() and 'web' in c.lower()][0]
y_col = [c for c in df.columns if 'str' in c.lower() or 'Str' in c][0]

b_data = df[b_col].values.astype(float)
H_data = df[H_col].values.astype(float)
y_data = df[y_col].values.astype(float)

B_BOUNDS = [(b_data.min() - 0.5, b_data.max() + 0.5)]
H_BOUNDS = [(H_data.min() - 0.5, H_data.max() + 0.5)]
BOUNDS_bH = [B_BOUNDS[0], H_BOUNDS[0]]

X_bH = np.column_stack([b_data, H_data])

print(f"Loaded {len(b_data)} beams")
print(f"b range: {b_data.min():.1f} to {b_data.max():.1f} mm")
print(f"H range: {H_data.min():.1f} to {H_data.max():.1f} mm")
print(f"Str/w range: {y_data.min():.1f} to {y_data.max():.1f} N/g")

## A. Look at the Data

Scatter plot of every beam your instructor tested. Color = strength-to-weight ratio.
The red star marks the strongest beam in the set.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(b_data, H_data, c=y_data, cmap='viridis', s=100, edgecolors='k', zorder=5)
best_idx = np.argmax(y_data)
ax.scatter(b_data[best_idx], H_data[best_idx], c='red', s=250, marker='*', zorder=10,
           label=f'Best: Str/w = {y_data[best_idx]:.1f}')
plt.colorbar(sc, label='Str/w (N/g)')
ax.set_xlabel('Web thickness b (mm)')
ax.set_ylabel('Web height H (mm)')
ax.set_title('Tested Beams')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Strongest beam: b={b_data[best_idx]:.2f}, H={H_data[best_idx]:.2f}, Str/w={y_data[best_idx]:.1f}")
print(f"Weakest beam:   b={b_data[np.argmin(y_data)]:.2f}, H={H_data[np.argmin(y_data)]:.2f}, Str/w={y_data.min():.1f}")

## B. Fit a Gaussian Process

A GP draws a smooth surface through your data points. At each point in the design space
it gives you two things: a prediction (its best guess for Str/w) and an uncertainty
(how confident it is in that guess).

Where data is close by, the GP is confident. Where the space is empty, uncertainty grows.

We work in log-space for Str/w because ratios are multiplicative, not additive. The GP
fits log(Str/w), then we convert back for plotting.

In [ ]:
# Train GP with a moderate noise level
alpha_default = np.sqrt(NOISE_LO * NOISE_HI)  # geometric mean of bounds
gp_bH, Xn_bH, yc_bH, ymean_bH = train_gp(X_bH, y_data, BOUNDS_bH, alpha=alpha_default)

# Predict on a grid
G0, G1, mu_grid, sig_grid = predict_grid(gp_bH, ymean_bH, BOUNDS_bH)

b_grid = G0 * (BOUNDS_bH[0][1] - BOUNDS_bH[0][0]) + BOUNDS_bH[0][0]
H_grid = G1 * (BOUNDS_bH[1][1] - BOUNDS_bH[1][0]) + BOUNDS_bH[1][0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicted Str/w
im0 = axes[0].contourf(b_grid, H_grid, mu_grid, levels=30, cmap='viridis')
axes[0].scatter(b_data, H_data, c='white', edgecolors='k', s=60, zorder=5)
plt.colorbar(im0, ax=axes[0], label='Predicted Str/w (N/g)')
axes[0].set_xlabel('b (mm)')
axes[0].set_ylabel('H (mm)')
axes[0].set_title('GP Mean Prediction')

# Uncertainty
im1 = axes[1].contourf(b_grid, H_grid, sig_grid, levels=30, cmap='Reds')
axes[1].scatter(b_data, H_data, c='white', edgecolors='k', s=60, zorder=5)
plt.colorbar(im1, ax=axes[1], label='Uncertainty (log-space std)')
axes[1].set_xlabel('b (mm)')
axes[1].set_ylabel('H (mm)')
axes[1].set_title('GP Uncertainty')

plt.tight_layout()
plt.show()

ls = gp_bH.kernel_.k2.length_scale
print(f"Learned length scales: b={ls[0]:.2f}, H={ls[1]:.2f}")
print("(Smaller = GP thinks that variable matters more)")

## C. Noise and Overfitting

Your measurements have noise. If you test the same beam twice you will not get the same
number. Your instructor tested duplicate beams and measured a noise floor.

The noise parameter (alpha) tells the GP how much to trust each data point. Too low and
the GP chases every bump in the data (overfitting). Too high and it ignores real patterns.

Below: the same data fit with three different noise levels. Watch how the surface changes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
alphas = [NOISE_LO * 0.1, np.sqrt(NOISE_LO * NOISE_HI), NOISE_HI]
titles = ['Too low (overfit)', 'Moderate', 'High (smooth)']

for ax, a, t in zip(axes, alphas, titles):
    gp_tmp, _, _, ym_tmp = train_gp(X_bH, y_data, BOUNDS_bH, alpha=a)
    G0t, G1t, mu_t, _ = predict_grid(gp_tmp, ym_tmp, BOUNDS_bH)
    bg = G0t * (BOUNDS_bH[0][1] - BOUNDS_bH[0][0]) + BOUNDS_bH[0][0]
    hg = G1t * (BOUNDS_bH[1][1] - BOUNDS_bH[1][0]) + BOUNDS_bH[1][0]
    im = ax.contourf(bg, hg, mu_t, levels=30, cmap='viridis')
    ax.scatter(b_data, H_data, c='white', edgecolors='k', s=50, zorder=5)
    plt.colorbar(im, ax=ax, label='Str/w')
    ax.set_xlabel('b (mm)')
    ax.set_ylabel('H (mm)')
    ax.set_title(f'{t}\nalpha={a:.1e}')

plt.tight_layout()
plt.show()

### Heteroscedastic Noise

Not all beams are tested under the same conditions. Beams near the lateral stability
boundary (R close to 1) tend to fail in less predictable ways. The noise depends on
the design.

R is the stability ratio: the ratio of buckling strength to yield strength. Below we
compute R for each beam and show how giving the GP location-dependent noise (heteroscedastic)
changes the fit compared to constant noise (homoscedastic).

In [ ]:
# Compute R for each beam
R_data = np.array([bH_to_R(b_data[i], H_data[i]) for i in range(len(b_data))])

# Heteroscedastic alpha: noise depends on R
alpha_hetero = np.array([get_noise_variance(R_data[i]) for i in range(len(R_data))])

# Train both versions
alpha_homo = np.sqrt(NOISE_LO * NOISE_HI)
gp_homo, _, _, ym_homo = train_gp(X_bH, y_data, BOUNDS_bH, alpha=alpha_homo)
gp_het, Xn_het, yc_het, ym_het = train_gp(X_bH, y_data, BOUNDS_bH, alpha=alpha_hetero)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# R vs Str/w
axes[0].scatter(R_data, y_data, c=b_data, cmap='coolwarm', s=80, edgecolors='k')
axes[0].set_xlabel('Stability ratio R')
axes[0].set_ylabel('Str/w (N/g)')
axes[0].set_title('R vs Str/w (color = b)')
cb = plt.colorbar(axes[0].collections[0], ax=axes[0])
cb.set_label('b (mm)')

# Homo surface
G0h, G1h, mu_homo, _ = predict_grid(gp_homo, ym_homo, BOUNDS_bH)
bg = G0h * (BOUNDS_bH[0][1] - BOUNDS_bH[0][0]) + BOUNDS_bH[0][0]
hg = G1h * (BOUNDS_bH[1][1] - BOUNDS_bH[1][0]) + BOUNDS_bH[1][0]
im1 = axes[1].contourf(bg, hg, mu_homo, levels=30, cmap='viridis')
axes[1].scatter(b_data, H_data, c='white', edgecolors='k', s=50, zorder=5)
plt.colorbar(im1, ax=axes[1], label='Str/w')
axes[1].set_title('Constant noise')
axes[1].set_xlabel('b (mm)')
axes[1].set_ylabel('H (mm)')

# Hetero surface
_, _, mu_het, _ = predict_grid(gp_het, ym_het, BOUNDS_bH)
im2 = axes[2].contourf(bg, hg, mu_het, levels=30, cmap='viridis')
axes[2].scatter(b_data, H_data, c='white', edgecolors='k', s=50, zorder=5)
plt.colorbar(im2, ax=axes[2], label='Str/w')
axes[2].set_title('R-dependent noise')
axes[2].set_xlabel('b (mm)')
axes[2].set_ylabel('H (mm)')

plt.tight_layout()
plt.show()

print("Look at how the surfaces differ, especially near thin webs (low b) where R is low.")

## D. Physics-Informed Features

So far the GP knows two things about each beam: web thickness (b) and web height (H).
We can give it more by computing physics-based features from b and H.

**dH** = how far this beam's web height is from the theoretically optimal H for its
web thickness. Positive means taller than optimal, negative means shorter.

**R** = stability ratio = M_cr / M_yield. This is the ratio of the critical buckling
moment to the yield moment. R > 1 means the beam yields before it buckles (stable).
R < 1 means it buckles first (unstable, bad).

A thinner web (low b) uses less material, so you might expect higher strength-to-weight.
But thin webs buckle sideways before they yield. R captures that tradeoff. Giving R to
the GP lets it learn the stability story from fewer data points.

In [ ]:
# Compute physics features for each beam
dH_data = np.array([bH_to_dH(b_data[i], H_data[i]) for i in range(len(b_data))])

print("Physics features for each beam:")
feat_df = pd.DataFrame({
    'b': np.round(b_data, 2),
    'H': np.round(H_data, 2),
    'dH': np.round(dH_data, 2),
    'R': np.round(R_data, 2),
    'Str/w': np.round(y_data, 1)
})
print(feat_df.to_string(index=False))

# 3D parameterization: (b, dH, R)
X_3d = np.column_stack([b_data, dH_data, R_data])
BOUNDS_3d = [
    (b_data.min() - 0.5, b_data.max() + 0.5),
    (dH_data.min() - 1.0, dH_data.max() + 1.0),
    (max(0, R_data.min() - 0.5), R_data.max() + 0.5),
]

gp_3d, Xn_3d, yc_3d, ym_3d = train_gp(X_3d, y_data, BOUNDS_3d, alpha=alpha_default)

ls_3d = gp_3d.kernel_.k2.length_scale
print(f"\n3D GP length scales: b={ls_3d[0]:.2f}, dH={ls_3d[1]:.2f}, R={ls_3d[2]:.2f}")
print("Compare to 2D GP length scales: b={:.2f}, H={:.2f}".format(
    gp_bH.kernel_.k2.length_scale[0], gp_bH.kernel_.k2.length_scale[1]))
print("\nSmaller length scale = GP thinks that variable matters more for prediction.")

In [ ]:
# Compare 2D vs 3D GP predictions at the same (b, H) points
# Generate candidates in (b, H), predict with both GPs
n_test = 2000
np.random.seed(0)
b_test = np.random.uniform(BOUNDS_bH[0][0], BOUNDS_bH[0][1], n_test)
H_test = np.random.uniform(BOUNDS_bH[1][0], BOUNDS_bH[1][1], n_test)

# 2D predictions
X_test_2d = np.column_stack([b_test, H_test])
Xt_2d_n = normalize(X_test_2d, BOUNDS_bH)
mu_2d, sig_2d = gp_bH.predict(Xt_2d_n, return_std=True)
mu_2d_real = np.exp(mu_2d + ymean_bH)

# 3D predictions
dH_test = np.array([bH_to_dH(b_test[i], H_test[i]) for i in range(n_test)])
R_test = np.array([bH_to_R(b_test[i], H_test[i]) for i in range(n_test)])
X_test_3d = np.column_stack([b_test, dH_test, R_test])
Xt_3d_n = normalize(X_test_3d, BOUNDS_3d)
mu_3d, sig_3d = gp_3d.predict(Xt_3d_n, return_std=True)
mu_3d_real = np.exp(mu_3d + ym_3d)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc0 = axes[0].scatter(b_test, H_test, c=sig_2d, cmap='Reds', s=10, alpha=0.6)
axes[0].scatter(b_data, H_data, c='black', s=60, zorder=5)
plt.colorbar(sc0, ax=axes[0], label='Uncertainty')
axes[0].set_xlabel('b (mm)')
axes[0].set_ylabel('H (mm)')
axes[0].set_title('2D GP (b, H) uncertainty')

sc1 = axes[1].scatter(b_test, H_test, c=sig_3d, cmap='Reds', s=10, alpha=0.6)
axes[1].scatter(b_data, H_data, c='black', s=60, zorder=5)
plt.colorbar(sc1, ax=axes[1], label='Uncertainty')
axes[1].set_xlabel('b (mm)')
axes[1].set_ylabel('H (mm)')
axes[1].set_title('3D GP (b, dH, R) uncertainty')

plt.tight_layout()
plt.show()
print("The 3D GP uses physics to be more confident in regions it understands, even without data there.")

## E. Pick Your Beam

You have a model. Now you need to decide where to test next. Two acquisition functions
can help:

**UCB (Upper Confidence Bound):**

    score = predicted_mean + kappa * predicted_uncertainty

High kappa means you value exploration (testing where you are uncertain). Low kappa
means you value exploitation (testing where the prediction is already high). kappa = 0
would just pick the predicted best. kappa = 5 would chase the biggest unknowns.

**EI (Expected Improvement):**

EI computes the expected gain over your current best, weighted by probability. It
balances exploration and exploitation in one formula. The parameter xi controls how
much improvement you require before exploring. Small xi is exploitative, large xi
is exploratory.

Below, set your choices. Your student ID seeds the random candidate generation, so
every student gets a unique set of candidates even with identical settings.

In [ ]:
# ============================================================
# YOUR CHOICES (edit these values)
# ============================================================
student_id = 0              # your student ID number (integer)
noise_alpha = 3e-4          # noise level, between 3.0e-05 and 3.0e-03
exploration = 2.0           # kappa (for UCB) or xi (for EI), range: 0.1 to 5.0
acquisition = 'ucb'         # 'ucb' or 'ei'
parameterization = 'b_dH_R' # 'b_H' or 'b_dH_R'
heteroscedastic = False     # True or False
# ============================================================

In [ ]:
np.random.seed(student_id * 1000 + 7)

# Train GP with student's choices
if heteroscedastic:
    R_train = np.array([bH_to_R(b_data[i], H_data[i]) for i in range(len(b_data))])
    alpha_arr = np.array([get_noise_variance(R_train[i]) for i in range(len(R_train))])
    # Scale heteroscedastic noise by student's alpha choice relative to baseline
    scale = noise_alpha / np.mean(alpha_arr)
    alpha_use = alpha_arr * scale
else:
    alpha_use = noise_alpha

if parameterization == 'b_dH_R':
    X_train = X_3d
    bounds_use = BOUNDS_3d
else:
    X_train = X_bH
    bounds_use = BOUNDS_bH

gp_stu, Xn_stu, yc_stu, ym_stu = train_gp(X_train, y_data, bounds_use, alpha=alpha_use)

# Generate candidates in (b, H) space, transform if needed
N_CAND = 15000
b_cand = np.random.uniform(BOUNDS_bH[0][0], BOUNDS_bH[0][1], N_CAND)
H_cand = np.random.uniform(BOUNDS_bH[1][0], BOUNDS_bH[1][1], N_CAND)

if parameterization == 'b_dH_R':
    dH_cand = np.array([bH_to_dH(b_cand[i], H_cand[i]) for i in range(N_CAND)])
    R_cand = np.array([bH_to_R(b_cand[i], H_cand[i]) for i in range(N_CAND)])
    X_cand = np.column_stack([b_cand, dH_cand, R_cand])
    X_cand_n = normalize(X_cand, BOUNDS_3d)
else:
    X_cand = np.column_stack([b_cand, H_cand])
    X_cand_n = normalize(X_cand, BOUNDS_bH)

mu_cand, sig_cand = gp_stu.predict(X_cand_n, return_std=True)

# Compute acquisition scores
if acquisition == 'ucb':
    scores = mu_cand + exploration * sig_cand
elif acquisition == 'ei':
    best_y = np.max(yc_stu)
    Z = np.where(sig_cand > 1e-9, (mu_cand - best_y - exploration) / sig_cand, 0.0)
    scores = np.where(sig_cand > 1e-9,
                      (mu_cand - best_y - exploration) * norm.cdf(Z) + sig_cand * norm.pdf(Z),
                      0.0)

# Top N recommendations
top_idx = np.argsort(scores)[::-1][:N_BEAMS_TO_PICK]

print("=" * 60)
print("YOUR RECOMMENDED BEAM(S)")
print("=" * 60)
for rank, idx in enumerate(top_idx):
    pred_strw = np.exp(mu_cand[idx] + ym_stu)
    unc = sig_cand[idx]
    print(f"  #{rank+1}: b = {b_cand[idx]:.2f} mm, H = {H_cand[idx]:.2f} mm")
    print(f"       Predicted Str/w = {pred_strw:.1f} N/g  (uncertainty = {unc:.3f})")
    print()

# Plot recommendation on the GP surface
fig, ax = plt.subplots(figsize=(8, 6))
# Plot the GP prediction in (b, H) space
G0p, G1p = np.meshgrid(np.linspace(0, 1, 80), np.linspace(0, 1, 80))
if parameterization == 'b_dH_R':
    bp = G0p.ravel() * (BOUNDS_bH[0][1] - BOUNDS_bH[0][0]) + BOUNDS_bH[0][0]
    Hp = G1p.ravel() * (BOUNDS_bH[1][1] - BOUNDS_bH[1][0]) + BOUNDS_bH[1][0]
    dHp = np.array([bH_to_dH(bp[i], Hp[i]) for i in range(len(bp))])
    Rp = np.array([bH_to_R(bp[i], Hp[i]) for i in range(len(bp))])
    Xp = normalize(np.column_stack([bp, dHp, Rp]), BOUNDS_3d)
else:
    Xp = np.column_stack([G0p.ravel(), G1p.ravel()])

mup, _ = gp_stu.predict(Xp, return_std=True)
mup_real = np.exp(mup + ym_stu)
bg = G0p * (BOUNDS_bH[0][1] - BOUNDS_bH[0][0]) + BOUNDS_bH[0][0]
hg = G1p * (BOUNDS_bH[1][1] - BOUNDS_bH[1][0]) + BOUNDS_bH[1][0]

im = ax.contourf(bg, hg, mup_real.reshape(G0p.shape), levels=30, cmap='viridis')
ax.scatter(b_data, H_data, c='white', edgecolors='k', s=60, zorder=5, label='Training data')
for rank, idx in enumerate(top_idx):
    ax.scatter(b_cand[idx], H_cand[idx], c='red', s=200, marker='*', zorder=10,
               label=f'Rec #{rank+1}' if rank == 0 else None)
plt.colorbar(im, label='Predicted Str/w (N/g)')
ax.set_xlabel('b (mm)')
ax.set_ylabel('H (mm)')
ax.set_title(f'Your GP ({parameterization}, {acquisition}, alpha={noise_alpha:.1e}, kappa/xi={exploration})')
ax.legend()
plt.tight_layout()
plt.show()

## F. Reflect

Write 2-3 sentences about why you chose these settings. Consider:
- Did you lean toward exploring unknown regions or exploiting what looks good?
- Did the physics parameterization change your recommendation?
- Did the noise level matter?

You are graded on reasoning, not on whether your beam is the strongest.

In [ ]:
print("=" * 50)
print("SUBMISSION SUMMARY")
print("=" * 50)
print(f"Student ID:       {student_id}")
print(f"Parameterization: {parameterization}")
print(f"Acquisition:      {acquisition}")
print(f"Exploration:      {exploration}")
print(f"Noise alpha:      {noise_alpha:.2e}")
print(f"Heteroscedastic:  {heteroscedastic}")
print()
for rank, idx in enumerate(top_idx):
    pred_strw = np.exp(mu_cand[idx] + ym_stu)
    print(f"Beam #{rank+1}: b = {b_cand[idx]:.2f} mm, H = {H_cand[idx]:.2f} mm, Predicted = {pred_strw:.1f} N/g")
print("=" * 50)